# A2.7 · Systems that don't understand agents

**Function A — Security Architecture & Platform → The Identity & Non-Human Identity Engineer**  ·  *Security of AI*

Builds on **[A2.6 · The agentic gateway](https://spbreed.github.io/cyber-commons/lessons/A2.6.html)**.

| | |
|---|---|
| Open-source tooling | agentgateway, OPA |
| Open-weight models | — |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


A2.6 assumed you can put a gateway in front of the system. Sometimes you cannot.

There is a category of system where the agent must be allowed access, the system
cannot be modified, and no meaningful per-agent identity is possible:

- A vendor SaaS with one API key per tenant.
- A mainframe or ERP where the integration account was configured in 2003 and
  the person who understood it has retired.
- A database that only does username/password, reached over a private link.

You cannot give these systems identity. What you *can* do is control **the one
place every call passes through** — a choke point — and apply the controls that
do not need identity:

- **Rate limiting**, so a compromised agent cannot drain a dataset at machine
  speed.
- **Volume and pattern budgets**, because the difference between an agent doing
  its job and an agent exfiltrating is almost always *quantity*.
- **Time-of-day and concurrency limits**, matching the business process the
  legacy system actually serves.
- **Full recording at the choke point**, since the downstream log is useless.

This is a genuinely weaker control than identity, and it should be labelled as
such rather than presented as equivalent. What it buys is a bounded worst case.

## 2 · Demo — the legacy system, and why identity is off the table

A customer database behind one shared account. Every consumer — three agents and a nightly batch job — presents the same credential.

In [ ]:
import time
from dataclasses import dataclass, field

LEGACY = {"name": "crm-oracle-prod", "auth": "shared username/password",
          "per_caller_identity": False, "can_be_modified": False,
          "holds": "1.2M customer records"}
for k, v in LEGACY.items():
    print(f"{k:22s} {v}")

CONSUMERS = ["support-agent", "billing-agent", "analytics-agent", "nightly-batch"]
print(f"\nconsumers sharing one credential: {CONSUMERS}")
print("the database's own log will attribute every query to 'SVC_CRM_INT'.")

## 3 · Where it breaks — normal use and abuse look identical

The support agent legitimately reads customer records. So does an agent whose prompt has been compromised. Per-query, they are indistinguishable — which is exactly why the control has to be about *rate and volume*, not about intent.

In [ ]:
def session(name, queries, rows_each, seconds):
    return {"caller": name, "queries": queries, "rows": queries * rows_each,
            "seconds": seconds, "rows_per_min": queries * rows_each / (seconds/60)}

sessions = [
    session("support-agent  (normal)",     40,     1, 3600),
    session("billing-agent  (normal)",    600,     1, 3600),
    session("nightly-batch  (normal)",      1, 50000, 1800),
    session("support-agent  (compromised)", 9000,  50,  120),
]
print(f"{'caller':30s}{'queries':>8}{'rows':>9}{'rows/min':>11}")
print("-" * 60)
for s in sessions:
    print(f"{s['caller']:30s}{s['queries']:>8}{s['rows']:>9}{s['rows_per_min']:>11.0f}")
print("\nEvery one of these presents the same credential and issues valid SQL.")
print("The compromised session is not doing anything the account may not do —")
print("it is doing a permitted thing far too much.")

## 4 · The control — a throttling choke point

One place all four consumers must pass through. It cannot tell them apart cryptographically, but it *can* give each a named lane with its own budget, and enforce the budget.

In [ ]:
@dataclass
class ChokePoint:
    """The one place every call to the legacy system passes through.

    Identity here is a *declared lane*, not a proven one — a compromised agent
    could claim another lane. That is an honest weakness: the control bounds
    the worst case, it does not attribute. Say so in the design doc.
    """
    budgets: dict                        # lane -> (rows_per_min, max_concurrent)
    window_start: float = field(default_factory=time.time)
    used: dict = field(default_factory=dict)
    log: list = field(default_factory=list)

    def query(self, lane, rows, at=None):
        at = at or time.time()
        limit, _ = self.budgets.get(lane, (0, 0))
        spent = self.used.get(lane, 0)
        if spent + rows > limit:
            entry = {"lane": lane, "rows": rows, "allowed": False,
                     "why": f"rate budget exceeded: {spent}+{rows} > {limit} rows/min"}
            self.log.append(entry); return entry
        self.used[lane] = spent + rows
        entry = {"lane": lane, "rows": rows, "allowed": True,
                 "remaining": limit - self.used[lane]}
        self.log.append(entry); return entry

choke = ChokePoint(budgets={
    "support-agent":   (200,    4),      # a human-paced support workflow
    "billing-agent":   (1200,   8),
    "analytics-agent": (5000,   2),
    "nightly-batch":   (60000,  1),      # bulk, but only in its window
})

print("normal traffic:")
for lane, rows in [("support-agent", 30), ("support-agent", 45),
                   ("billing-agent", 900), ("nightly-batch", 50000)]:
    r = choke.query(lane, rows)
    print(f"   {lane:18s} {rows:>6} rows → "
          f"{'ok, ' + str(r['remaining']) + ' left' if r['allowed'] else r['why']}")

print("\ncompromised support agent tries to drain the table:")
for attempt in range(1, 5):
    r = choke.query("support-agent", 5000)
    print(f"   attempt {attempt}: {'ALLOWED' if r['allowed'] else 'BLOCKED — ' + r['why']}")

In [ ]:
# Verify: what did the choke point actually bound?
attacker_got = sum(e["rows"] for e in choke.log
                   if e["lane"] == "support-agent" and e["allowed"])
unbounded = 9000 * 50
print(f"rows the compromised lane obtained: {attacker_got}")
print(f"rows it would have obtained unthrottled: {unbounded:,}")
print(f"reduction: {unbounded / max(attacker_got,1):,.0f}×")
print("\nHonest framing for the design doc:")
print("  · this does NOT tell you which agent misbehaved (no identity available)")
print("  · it DOES bound the worst case to one lane's budget per window")
print("  · and the choke point log is the only per-caller record that exists")
assert attacker_got < unbounded

## What you just proved

The legacy system is shown with no per-caller identity. Normal and compromised sessions are indistinguishable per query — the compromised one differs only by rate (225,000 rows/min vs 40). The choke point permits normal traffic and blocks the drain attempts, bounding what the attacker obtains to a fraction of the unthrottled 450,000 rows.

## Your turn

Find one system in your estate where every agent shares a credential. Set a rows-per-minute budget from the *legitimate* workload's 99th percentile, not from the system's capacity. The gap between those two numbers is what an attacker currently has.

---

**Next → [A2.8 · Just-in-time authority](https://spbreed.github.io/cyber-commons/lessons/A2.8.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A2.7.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A2.7.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*